In [1]:
!pip install -q thop

In [2]:
import tensorflow as tf
import gc

tf.keras.backend.clear_session()
gc.collect()

0

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
%cd /content/drive/MyDrive

/content/drive/MyDrive


In [5]:
import os
import gc
import copy
import time
import random
import shutil

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.preprocessing import label_binarize

from thop import profile

# RANDOM SEED

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# DEVICE

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("WARNING: GPU not available.")

Device: cuda
GPU: Tesla T4
GPU Memory: 14.56 GB


In [6]:
DATASET_DIR = "/content/drive/MyDrive/MANGO/Public Data (Mango)"

In [7]:
# OUTPUT DIRECTORY

OUTPUT_DIR = "/content/drive/MyDrive/MANGO"

SPLIT_DIR = "/content/drive/MyDrive/MANGO/mango_split"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

os.makedirs(
    SPLIT_DIR,
    exist_ok=True
)

In [8]:
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

TRAIN_DIR = os.path.join(
    SPLIT_DIR,
    "train"
)

VAL_DIR = os.path.join(
    SPLIT_DIR,
    "val"
)

TEST_DIR = os.path.join(
    SPLIT_DIR,
    "test"
)

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

# GET CLASSES
classes = sorted([
    x for x in os.listdir(DATASET_DIR)
    if os.path.isdir(
        os.path.join(DATASET_DIR, x)
    )
])

print("Classes:")
print(classes)

print("Number of classes:", len(classes))

# SPLIT EACH CLASS

summary = []


for class_name in classes:

    source_dir = os.path.join(
        DATASET_DIR,
        class_name
    )

    images = [
        x for x in os.listdir(source_dir)
        if x.lower().endswith(
            (
                ".jpg",
                ".jpeg",
                ".png",
                ".bmp",
                ".webp"
            )
        )
    ]

    print(
        f"{class_name}: {len(images)} images"
    )

    # 70% TRAIN / 30% TEMP

    train_files, temp_files = train_test_split(
        images,
        test_size=0.30,
        random_state=SEED,
        shuffle=True
    )

    # 15% VALIDATION / 15% TEST

    val_files, test_files = train_test_split(
        temp_files,
        test_size=0.50,
        random_state=SEED,
        shuffle=True
    )

    # CREATE DIRECTORIES

    train_class = os.path.join(
        TRAIN_DIR,
        class_name
    )

    val_class = os.path.join(
        VAL_DIR,
        class_name
    )

    test_class = os.path.join(
        TEST_DIR,
        class_name
    )

    os.makedirs(
        train_class,
        exist_ok=True
    )

    os.makedirs(
        val_class,
        exist_ok=True
    )

    os.makedirs(
        test_class,
        exist_ok=True
    )


    # COPY FILES

    for filename in train_files:

        shutil.copy2(
            os.path.join(
                source_dir,
                filename
            ),
            os.path.join(
                train_class,
                filename
            )
        )


    for filename in val_files:

        shutil.copy2(
            os.path.join(
                source_dir,
                filename
            ),
            os.path.join(
                val_class,
                filename
            )
        )


    for filename in test_files:

        shutil.copy2(
            os.path.join(
                source_dir,
                filename
            ),
            os.path.join(
                test_class,
                filename
            )
        )

    summary.append({

        "Class": class_name,

        "Total": len(images),

        "Train": len(train_files),

        "Validation": len(val_files),

        "Test": len(test_files)

    })

summary_df = pd.DataFrame(summary)

display(summary_df)

print("\nTotal:", summary_df["Total"].sum())

print("Train:",  summary_df["Train"].sum())

print("Validation:", summary_df["Validation"].sum())

print("Test:", summary_df["Test"].sum())

Classes:
['Anthracnose', 'Bacterial Canker', 'Cutting Weevil', 'Die Back', 'Gall Midge', 'Healthy', 'Powdery Mildew', 'Sooty Mould']
Number of classes: 8
Anthracnose: 500 images
Bacterial Canker: 510 images
Cutting Weevil: 500 images
Die Back: 500 images
Gall Midge: 500 images
Healthy: 500 images
Powdery Mildew: 500 images
Sooty Mould: 500 images


,Class,Total,Train,Validation,Test
0,Anthracnose,500,350,75,75
1,Bacterial Canker,510,357,76,77
2,Cutting Weevil,500,350,75,75
3,Die Back,500,350,75,75
4,Gall Midge,500,350,75,75
5,Healthy,500,350,75,75
6,Powdery Mildew,500,350,75,75
7,Sooty Mould,500,350,75,75



Total: 4010
Train: 2807
Validation: 601
Test: 602


In [9]:
IMAGE_SIZE = 224

BATCH_SIZE = 4
# TRAIN TRANSFORM

train_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])


# VALIDATION / TEST TRANSFORM

test_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])


# DATASETS

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=test_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=test_transform
)

# DATA LOADERS

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


class_names = train_dataset.classes

num_classes = len(class_names)


print(
    "Classes:",
    class_names
)

print(
    "Number of classes:",
    num_classes
)

print(
    "Training:",
    len(train_dataset)
)

print(
    "Validation:",
    len(val_dataset)
)

print(
    "Testing:",
    len(test_dataset)
)

Classes: ['Anthracnose', 'Bacterial Canker', 'Cutting Weevil', 'Die Back', 'Gall Midge', 'Healthy', 'Powdery Mildew', 'Sooty Mould']
Number of classes: 8
Training: 2807
Validation: 601
Testing: 602


**CREATE** **MODELS**

In [10]:
def create_model(
    model_name,
    num_classes
):

    if model_name == "ConvNeXt":

        model = models.convnext_base(
            weights=models.ConvNeXt_Base_Weights.DEFAULT
        )

        in_features = (
            model.classifier[2].in_features
        )

        model.classifier[2] = nn.Linear(
            in_features,
            num_classes
        )


    elif model_name == "ViT-B/16":

        model = models.vit_b_16(
            weights=models.ViT_B_16_Weights.DEFAULT
        )

        in_features = (
            model.heads.head.in_features
        )

        model.heads.head = nn.Linear(
            in_features,
            num_classes
        )


    elif model_name == "Swin Transformer":

        model = models.swin_b(
            weights=models.Swin_B_Weights.DEFAULT
        )

        in_features = (
            model.head.in_features
        )

        model.head = nn.Linear(
            in_features,
            num_classes
        )


    else:

        raise ValueError(
            "Unknown model name"
        )


    return model.to(device)

In [11]:
def train_model(
    model,
    epochs=50
):

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )


    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs
    )

    # MIXED PRECISION

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda")
    )


    history = {

        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []

    }


    best_val_acc = 0.0

    best_weights = copy.deepcopy(
        model.state_dict()
    )


    for epoch in range(epochs):

        print(
            f"\nEpoch {epoch + 1}/{epochs}"
        )

        # TRAIN

        model.train()

        train_loss = 0.0

        train_correct = 0

        train_total = 0


        for images, labels in train_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):

                outputs = model(images)

                loss = criterion(
                    outputs,
                    labels
                )


            scaler.scale(
                loss
            ).backward()


            scaler.step(
                optimizer
            )


            scaler.update()


            train_loss += (
                loss.item() *
                images.size(0)
            )


            predictions = outputs.argmax(
                dim=1
            )


            train_correct += (
                predictions == labels
            ).sum().item()


            train_total += labels.size(0)


            # Free batch memory

            del images
            del labels
            del outputs
            del loss


        train_loss /= len(
            train_dataset
        )

        train_acc = (
            train_correct /
            train_total
        )

        # VALIDATION

        model.eval()

        val_loss = 0.0

        val_correct = 0

        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(
                    device,
                    non_blocking=True
                )

                labels = labels.to(
                    device,
                    non_blocking=True
                )


                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=(device.type == "cuda")
                ):

                    outputs = model(
                        images
                    )

                    loss = criterion(
                        outputs,
                        labels
                    )


                val_loss += (
                    loss.item() *
                    images.size(0)
                )


                predictions = outputs.argmax(
                    dim=1
                )


                val_correct += (
                    predictions == labels
                ).sum().item()


                val_total += labels.size(0)


        val_loss /= len(
            val_dataset
        )

        val_acc = (
            val_correct /
            val_total
        )


        scheduler.step()

        # SAVE HISTORY

        history["train_loss"].append(
            train_loss
        )

        history["val_loss"].append(
            val_loss
        )

        history["train_acc"].append(
            train_acc
        )

        history["val_acc"].append(
            val_acc
        )

        # SAVE BEST MODEL

        if val_acc > best_val_acc:

            best_val_acc = val_acc

            best_weights = copy.deepcopy(
                model.state_dict()
            )


        print(
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )


        # Free unused GPU memory

        if device.type == "cuda":

            torch.cuda.empty_cache()


        gc.collect()


    model.load_state_dict(
        best_weights
    )


    return model, history

In [12]:
def evaluate_model(
    model
):

    model.eval()

    y_true = []

    y_pred = []

    y_prob = []


    total_time = 0.0


    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(
                device,
                non_blocking=True
            )


            if device.type == "cuda":
                torch.cuda.synchronize()


            start = time.perf_counter()


            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):

                outputs = model(
                    images
                )


            if device.type == "cuda":
                torch.cuda.synchronize()


            end = time.perf_counter()


            total_time += (
                end - start
            )


            probabilities = torch.softmax(
                outputs,
                dim=1
            )


            predictions = outputs.argmax(
                dim=1
            )


            y_true.extend(
                labels.cpu().numpy()
            )

            y_pred.extend(
                predictions.cpu().numpy()
            )

            y_prob.extend(
                probabilities.cpu().numpy()
            )


    y_true = np.array(y_true)

    y_pred = np.array(y_pred)

    y_prob = np.array(y_prob)

    # METRICS

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    # ROC-AUC

    y_true_binary = label_binarize(
        y_true,
        classes=np.arange(
            num_classes
        )
    )


    try:

        roc_auc = roc_auc_score(
            y_true_binary,
            y_prob,
            multi_class="ovr",
            average="weighted"
        )

    except:

        roc_auc = np.nan

    # CONFUSION MATRIX

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    # INFERENCE TIME

    ms_per_image = (
        total_time /
        len(test_dataset)
    ) * 1000


    return {

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1-score": f1,

        "ROC-AUC": roc_auc,

        "Inference Time (ms/image)":
            ms_per_image

    }, cm, y_true, y_pred, y_prob

In [13]:
def plot_history(
    history,
    model_name
):

    epochs = range(
        1,
        len(
            history["train_loss"]
        ) + 1
    )

    # LOSS

    plt.figure(
        figsize=(7, 5)
    )

    plt.plot(
        epochs,
        history["train_loss"],
        label="Training Loss"
    )

    plt.plot(
        epochs,
        history["val_loss"],
        label="Validation Loss"
    )

    plt.xlabel("Epoch")

    plt.ylabel("Loss")

    plt.title(
        model_name +
        " Training/Validation Loss"
    )

    plt.legend()

    plt.grid(alpha=0)

    plt.show()

    # ACCURACY

    plt.figure(
        figsize=(7, 5)
    )

    plt.plot(
        epochs,
        history["train_acc"],
        label="Training Accuracy"
    )

    plt.plot(
        epochs,
        history["val_acc"],
        label="Validation Accuracy"
    )

    plt.xlabel("Epoch")

    plt.ylabel("Accuracy")

    plt.title(
        model_name +
        " Training/Validation Accuracy"
    )

    plt.legend()

    plt.grid(alpha=0)

    plt.show()

In [14]:
def plot_history(history, model_name):

    epochs = range(
        1,
        len(
            history["train_loss"]
        ) + 1
    )

    # LOSS

    plt.figure(
        figsize=(7, 5)
    )

    plt.plot(
        epochs,
        history["train_loss"],
        label="Training Loss"
    )

    plt.plot(
        epochs,
        history["val_loss"],
        label="Validation Loss"
    )

    plt.xlabel("Epoch")

    plt.ylabel("Loss")

    plt.title(
        model_name +
        " Training/Validation Loss"
    )

    plt.legend()

    plt.grid(alpha=0)

    plt.show()

    # ACCURACY

    plt.figure(
        figsize=(7, 5)
    )

    plt.plot(
        epochs,
        history["train_acc"],
        label="Training Accuracy"
    )

    plt.plot(
        epochs,
        history["val_acc"],
        label="Validation Accuracy"
    )

    plt.xlabel("Epoch")

    plt.ylabel("Accuracy")

    plt.title(
        model_name +
        " Training/Validation Accuracy"
    )

    plt.legend()

    plt.grid(alpha=0)
    plt.show()

**ROC**-**AUC** **CURVE**

In [15]:
# ROC-AUC CURVE FOR ALL MODELS

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

def plot_roc_auc_all_models(roc_data, class_names):

    plt.figure(figsize=(10, 8))

    for model_name, data in roc_data.items():

        y_true = data["y_true"]
        y_prob = data["y_prob"]

        # Convert labels to one-vs-rest format
        y_true_bin = label_binarize(
            y_true,
            classes=np.arange(len(class_names))
        )

        # Store all FPR and TPR values
        all_fpr = np.linspace(0, 1, 1000)
        mean_tpr = np.zeros_like(all_fpr)

        for i in range(len(class_names)):

            fpr, tpr, _ = roc_curve(
                y_true_bin[:, i],
                y_prob[:, i]
            )

            mean_tpr += np.interp(
                all_fpr,
                fpr,
                tpr
            )

        # Calculate mean ROC curve
        mean_tpr /= len(class_names)

        # Calculate AUC
        mean_auc = auc(
            all_fpr,
            mean_tpr
        )

        plt.plot(
            all_fpr,
            mean_tpr,
            linewidth=2,
            label=f"{model_name} (AUC = {mean_auc:.4f})"
        )

    # Random classifier line
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Random Classifier"
    )

    plt.xlabel("False Positive Rate", fontsize=12)

    plt.ylabel("True Positive Rate",fontsize=12)

    plt.title("ROC-AUC Curves for ConvNeXt, ViT-B/16 and Swin Transformer Models",fontsize=14, fontweight="bold")

    plt.legend(loc="lower right", fontsize=10)

    plt.grid(alpha=0)

    plt.xlim(0, 1)
    plt.ylim(0, 1.05)

    plt.tight_layout()

    plt.show()

**MODEL** **TRAINING**

**REAL**

In [16]:
# CONFUSION MATRIX PLOTTING FUNCTION

def plot_cm(cm, model_name):

    plt.figure(figsize=(10, 8))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.title(
        f"Confusion Matrix - {model_name}",
        fontsize=14
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")

    plt.xticks(
        rotation=45,
        ha="right"
    )

    plt.yticks(
        rotation=0
    )

    plt.tight_layout()

    # Save figure
    filename = (
        model_name
        .replace("/", "_")
        .replace(" ", "_")
        + "_confusion_matrix.png"
    )

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            filename
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    plt.close()

In [ ]:
# TRAIN AND EVALUATE ALL MODELS

MODEL_NAMES = ["ConvNeXt", "ViT-B/16", "Swin Transformer"]

EPOCHS = 2
roc_data = {}
results = []
histories = {}

for model_name in MODEL_NAMES:

    print("\n")
    print("=" * 70)
    print(f"TRAINING: {model_name}")
    print("=" * 70)

    # CLEAR MEMORY BEFORE CREATING A NEW MODEL

gc.collect()

if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    # CREATE MODEL

print(f"\nCreating {model_name} model...")

model = create_model(model_name=model_name,num_classes=num_classes)

    # Move model to CPU/GPU
model = model.to(device)

print(f"{model_name} model created successfully.")

    # TRAIN MODEL

print(f"\nStarting training for {model_name}...")

model, history = train_model(model=model,epochs=EPOCHS)

    # Save training history
histories[model_name] = history

    # EVALUATION

print(f"\nEvaluating {model_name}...")

model.eval()

metrics, cm, y_true, y_pred, y_prob = evaluate_model(model=model)

# SAVE ROC DATA FOR THIS MODEL
roc_data[model_name] = {"y_true": y_true, "y_prob": y_prob}
    # Add model name to metrics
metrics["Model"] = model_name

    # Store results
results.append(metrics)

    # PRINT RESULTS

print("\n")
print("-" * 50)
print(f"RESULTS FOR {model_name}")
print("-" * 50)

for key, value in metrics.items():

        if key != "Model":

            if isinstance(value, (int, float)):

                print(f"{key}: {value:.4f}")

            else:

                print(f"{key}: {value}")

    # CONFUSION MATRIX

print("\nGenerating confusion matrix...")

plot_cm(cm=cm, model_name=model_name)

    # TRAINING CURVES

print("Generating training curves...")

plot_history(history=history,  model_name=model_name)

    # CLASSIFICATION REPORT

print(f"\n Classification Report for {model_name} model")

print(classification_report(y_true, y_pred, target_names=class_names,digits=4,zero_division=0))

    # SAVE MODEL

filename = (
        model_name
        .replace("/", "_")
        .replace(" ", "_")
        + ".pth"
    )

model_path = os.path.join(OUTPUT_DIR, filename)

torch.save(model.state_dict(),model_path)

print(f"\nModel saved to:")
print(model_path)

    # DELETE MODEL FROM MEMORY

del model

gc.collect()

if torch.cuda.is_available():

        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

        print("GPU memory cleared.")

print("=" * 70)
print(f"FINISHED: {model_name}")
print("=" * 70)

# FINAL RESULTS TABLE

results_df = pd.DataFrame(results)

print("\n")
print("=" * 70)
print("FINAL MODEL COMPARISON")
print("=" * 70)

print(results_df.to_string(index=False))

# Save results
results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "model_comparison_results.csv"
    ),
    index=False
)

print("\nResults saved successfully.")



TRAINING: ConvNeXt


TRAINING: ViT-B/16


TRAINING: Swin Transformer

Creating Swin Transformer model...
Swin Transformer model created successfully.

Starting training for Swin Transformer...

Epoch 1/2


**PLot** **ROC**-**AUC**

In [ ]:
# PLOT ROC-AUC CURVES
plot_roc_auc_all_models(roc_data,class_names)

**Computational** **cost**

In [ ]:
def get_model_cost(model_name):
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    model = create_model(model_name,  num_classes )
    model.eval()
    # PARAMETERS
    parameters = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
    # FLOPs
    dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)

    try:
        flops, _ = profile(model,
            inputs=(dummy,),
            verbose=False
        )

    except Exception as e:

        print("FLOPs error:",e)
        flops = np.nan
    del dummy
    del model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return (parameters, trainable,flops)

**Calculate** **parameters** **and** **FLOPs**

In [ ]:
cost_results = []
for model_name in MODEL_NAMES:
    print("Calculating:", model_name)
    parameters, trainable, flops = (get_model_cost(model_name))
    cost_results.append({
        "Model": model_name,
        "Parameters": parameters,
        "Trainable Parameters": trainable,
        "FLOPs": flops,
        "Parameters (M)":
            parameters / 1_000_000,
        "FLOPs (G)":
            flops / 1_000_000_000
    })
cost_df = pd.DataFrame(cost_results)
display(cost_df.round(3))

**Final** **results** **table**

In [ ]:
results_df = pd.DataFrame(
    results
)

# MERGE COMPUTATIONAL RESULTS

results_df = results_df.merge(
    cost_df[
        [
            "Model",
            "Parameters",
            "Trainable Parameters",
            "FLOPs",
            "Parameters (M)",
            "FLOPs (G)"
        ]
    ],
    on="Model"
)

# ORDER COLUMNS

results_df = results_df[
    [

        "Model",

        "Accuracy",

        "Precision",

        "Recall",

        "F1-score",

        "ROC-AUC",

        "Parameters (M)",

        "FLOPs (G)",

        "Inference Time (ms/image)"

    ]
]

print(
    "FINAL RESULTS"
)

display(
    results_df.round(4)
)

**Save** **results**

In [ ]:
results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "mango_model_results.csv"
    ),
    index=False
)

print(
    "Results saved to:",
    OUTPUT_DIR
)